# Model Training & Evaluation (Leakage-Free)

In this notebook, we load the pre-split, preprocessed training and test datasets. We then train and evaluate several models (Logistic Regression, Random Forest, XGBoost) to predict customer churn. Finally, we show how to bundle the preprocessing and classifier into a single scikit-learn `Pipeline` for leakage-free production deployment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

### Step 1: Load Preprocessed Train and Test Data
These datasets were preprocessed using statistics computed strictly from the training set, avoiding any data leakage.

In [ ]:
train_df = pd.read_csv("../data/processed/train_cleaned.csv")
test_df = pd.read_csv("../data/processed/test_cleaned.csv")

X_train = train_df.drop("Churn", axis=1)
y_train = train_df["Churn"]

X_test = test_df.drop("Churn", axis=1)
y_test = test_df["Churn"]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

### Step 2: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy_lr)
print(classification_report(y_test, y_pred_lr))

### Step 3: Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", accuracy_rf)
print(classification_report(y_test, y_pred_rf))

### Step 4: XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])

print("XGBoost Accuracy:", accuracy_xgb)
print("XGBoost ROC-AUC:", roc_auc_xgb)
print(classification_report(y_test, y_pred_xgb))

### Step 5: Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=X_train.columns)
importance.nlargest(10).sort_values().plot(kind='barh')
plt.title("Top 10 Important Features (XGBoost)")
plt.show()

### Step 6: Leakage-Free Production Pipeline
To deploy the model to production safely, we bundle all preprocessing stages and the estimator into a single scikit-learn Pipeline. This allows the model to receive raw data, process it dynamically (using statistics learned from training data), and run predictions.

In [ ]:
import sys
sys.path.append("../src")
from preprocess import clean_raw_data, get_preprocessor
from sklearn.pipeline import Pipeline

# 1. Load the raw dataset
raw_df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
raw_cleaned = clean_raw_data(raw_df)

# 2. Split before fitting pipeline
X_raw = raw_cleaned.drop("Churn", axis=1)
y_raw = raw_cleaned["Churn"]
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 3. Build and fit pipeline
preprocessor = get_preprocessor()
prod_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

prod_pipeline.fit(X_train_raw, y_train_raw)

# 4. Save pipeline to pkl file
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(prod_pipeline, "../models/churn_model.pkl")
print("Production model pipeline serialized successfully to '../models/churn_model.pkl'!")